In [0]:
%run "/Workspace/Local To databrick manish migration/Resources/Dev/Con_note"

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

#calculation for customer mart
#find out the customer total purchase every month
#write the data into MySQL table
def customer_mart_calculation_table_write(final_customer_data_mart_df):
    window = Window.partitionBy("customer_id","sales_date_month")
    final_customer_data_mart = final_customer_data_mart_df.withColumn("sales_date_month",
                                           substring(col("sales_date"),1,7))\
                    .withColumn("total_sales_every_month_by_each_customer",
                                sum("total_cost").over(window))\
                    .select("customer_id", concat(col("customer_first_name"),lit(" "),col("customer_last_name"))
                            .alias("full_name"),"customer_address","customer_pincode",
                            "sales_date_month",
                            col("total_sales_every_month_by_each_customer").alias("total_sales"))\
                    .distinct()

    final_customer_data_mart.show()
    final_customer_data_mart.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true")\
    .saveAsTable("youtube_project.youtube_project_database.customers_data_mart")


